# **N1-1 – Orquestrando Transformações SQL**

Continuação do N1-Analytics. As tabelas são as mesmas (clientes, categorias, produtos e pedidos), mas agora o grafo não mostra a relação entre as entidades e sim o **fluxo das transformações SQL**: cada nó é uma etapa do pipeline e cada aresta indica que uma etapa precisa do resultado de outra antes de rodar.

In [ ]:
!pip install pyvis networkx pandas matplotlib

In [ ]:
import sqlite3
import time
import copy
import pandas as pd
import IPython
import networkx as nx
import matplotlib.pyplot as plt
from pyvis.network import Network

# **Etapas do Pipeline em SQL**

Cada etapa guarda o SQL que executa e a lista de etapas de que depende. Usei o SQLite (já vem no Python) no lugar do Databricks para conseguir executar tudo direto no Colab.

In [ ]:
etapas = {}

# 1. Carga das tabelas de origem (mesmos dados do N1, com mais alguns registros)
etapas["clientes"] = {
    "tipo": "carga",
    "depende_de": [],
    "descricao": "Cria e carrega a tabela de clientes",
    "sql": """
        DROP TABLE IF EXISTS clientes;
        CREATE TABLE clientes (
            cliente_id INT,
            nome TEXT,
            email TEXT,
            cidade TEXT
        );
        INSERT INTO clientes VALUES
        (1, 'Ana Silva', 'ana@gmail.com', 'Lisboa'),
        (2, 'Bruno Santos', 'bruno@gmail.com', 'Porto'),
        (3, 'Carla Mendes', 'carla@gmail.com', 'Coimbra'),
        (4, 'Diego Rocha', 'diego@gmail.com', 'Braga'),
        (5, 'Elisa Costa', 'elisa@gmail.com', 'Lisboa');
    """
}

etapas["categorias"] = {
    "tipo": "carga",
    "depende_de": [],
    "descricao": "Cria e carrega a tabela de categorias",
    "sql": """
        DROP TABLE IF EXISTS categorias;
        CREATE TABLE categorias (
            categoria_id INT,
            nome_categoria TEXT,
            departamento TEXT
        );
        INSERT INTO categorias VALUES
        (10, 'Eletrônicos', 'Tecnologia'),
        (20, 'Vestuário', 'Moda'),
        (30, 'Livros', 'Cultura'),
        (40, 'Calçados', 'Moda');
    """
}

etapas["produtos"] = {
    "tipo": "carga",
    "depende_de": [],
    "descricao": "Cria e carrega a tabela de produtos",
    "sql": """
        DROP TABLE IF EXISTS produtos;
        CREATE TABLE produtos (
            produto_id INT,
            nome_produto TEXT,
            categoria_id INT,
            preco DECIMAL(10,2)
        );
        INSERT INTO produtos VALUES
        (100, 'Smartphone', 10, 500.00),
        (101, 'Fone de Ouvido', 10, 80.00),
        (102, 'Notebook', 10, 1200.00),
        (200, 'Camisa', 20, 25.00),
        (201, 'Calça Jeans', 20, 60.00),
        (300, 'Livro de SQL', 30, 45.00),
        (400, 'Tênis', 40, 120.00);
    """
}

# o pedido 1008 é de um cliente que não existe e o 1010 está com quantidade zerada
etapas["pedidos"] = {
    "tipo": "carga",
    "depende_de": [],
    "descricao": "Cria e carrega a tabela de pedidos",
    "sql": """
        DROP TABLE IF EXISTS pedidos;
        CREATE TABLE pedidos (
            pedido_id INT,
            cliente_id INT,
            produto_id INT,
            data_pedido DATE,
            quantidade INT,
            valor_total DECIMAL(10,2)
        );
        INSERT INTO pedidos VALUES
        (1000, 1, 100, '2026-01-15', 1, 500.00),
        (1001, 2, 200, '2026-01-16', 2, 50.00),
        (1002, 3, 102, '2026-01-16', 1, 1200.00),
        (1003, 1, 101, '2026-01-17', 2, 160.00),
        (1004, 4, 400, '2026-01-17', 1, 120.00),
        (1005, 5, 300, '2026-01-18', 3, 135.00),
        (1006, 2, 100, '2026-01-18', 1, 500.00),
        (1007, 3, 201, '2026-01-19', 2, 120.00),
        (1008, 6, 200, '2026-01-19', 1, 25.00),
        (1009, 4, 102, '2026-01-20', 1, 1200.00),
        (1010, 5, 200, '2026-01-20', 0, 0.00),
        (1011, 1, 400, '2026-01-21', 2, 240.00),
        (1012, 2, 300, '2026-01-21', 1, 45.00),
        (1013, 3, 100, '2026-01-22', 1, 500.00),
        (1014, 5, 102, '2026-01-22', 1, 1200.00);
    """
}

In [ ]:
# 2. Limpeza e validação
etapas["produtos_completos"] = {
    "tipo": "transformacao",
    "depende_de": ["produtos", "categorias"],
    "descricao": "Junta cada produto com a sua categoria e departamento",
    "sql": """
        DROP TABLE IF EXISTS produtos_completos;
        CREATE TABLE produtos_completos AS
        SELECT p.produto_id, p.nome_produto, p.preco,
               c.categoria_id, c.nome_categoria, c.departamento
        FROM produtos p
        JOIN categorias c ON p.categoria_id = c.categoria_id;
    """
}

etapas["pedidos_validos"] = {
    "tipo": "transformacao",
    "depende_de": ["pedidos", "clientes", "produtos"],
    "descricao": "Remove pedidos com quantidade zerada ou com cliente/produto que não existe",
    "sql": """
        DROP TABLE IF EXISTS pedidos_validos;
        CREATE TABLE pedidos_validos AS
        SELECT *
        FROM pedidos
        WHERE quantidade > 0
          AND cliente_id IN (SELECT cliente_id FROM clientes)
          AND produto_id IN (SELECT produto_id FROM produtos);
    """
}

# etapa de verificação: o SELECT lista os problemas, se voltar alguma linha a etapa falha
etapas["verificacao_pedidos"] = {
    "tipo": "verificacao",
    "depende_de": ["pedidos_validos", "produtos"],
    "descricao": "Confere se o valor_total bate com quantidade x preço e se não tem pedido duplicado",
    "sql": """
        SELECT pv.pedido_id, 'valor_total diferente de quantidade x preço' AS problema
        FROM pedidos_validos pv
        JOIN produtos pr ON pv.produto_id = pr.produto_id
        WHERE ROUND(pv.quantidade * pr.preco, 2) <> pv.valor_total
        UNION ALL
        SELECT pedido_id, 'pedido duplicado' AS problema
        FROM pedidos_validos
        GROUP BY pedido_id
        HAVING COUNT(*) > 1
    """
}

In [ ]:
# 3. Tabela de vendas e agregações
etapas["vendas_detalhadas"] = {
    "tipo": "transformacao",
    "depende_de": ["pedidos_validos", "clientes", "produtos_completos", "verificacao_pedidos"],
    "descricao": "Junta pedido, cliente, produto e categoria em uma tabela só (só roda se a verificação passar)",
    "sql": """
        DROP TABLE IF EXISTS vendas_detalhadas;
        CREATE TABLE vendas_detalhadas AS
        SELECT pv.pedido_id, pv.data_pedido,
               c.cliente_id, c.nome AS cliente, c.cidade,
               pc.nome_produto, pc.nome_categoria, pc.departamento,
               pv.quantidade, pv.valor_total
        FROM pedidos_validos pv
        JOIN clientes c ON pv.cliente_id = c.cliente_id
        JOIN produtos_completos pc ON pv.produto_id = pc.produto_id;
    """
}

etapas["vendas_por_categoria"] = {
    "tipo": "transformacao",
    "depende_de": ["vendas_detalhadas"],
    "descricao": "Faturamento e unidades vendidas por categoria",
    "sql": """
        DROP TABLE IF EXISTS vendas_por_categoria;
        CREATE TABLE vendas_por_categoria AS
        SELECT nome_categoria, departamento,
               COUNT(*) AS total_pedidos,
               SUM(quantidade) AS unidades,
               SUM(valor_total) AS faturamento
        FROM vendas_detalhadas
        GROUP BY nome_categoria, departamento
        ORDER BY faturamento DESC;
    """
}

etapas["vendas_por_cliente"] = {
    "tipo": "transformacao",
    "depende_de": ["vendas_detalhadas"],
    "descricao": "Total de pedidos, faturamento e ticket médio por cliente",
    "sql": """
        DROP TABLE IF EXISTS vendas_por_cliente;
        CREATE TABLE vendas_por_cliente AS
        SELECT cliente, cidade,
               COUNT(*) AS total_pedidos,
               SUM(valor_total) AS faturamento,
               ROUND(AVG(valor_total), 2) AS ticket_medio
        FROM vendas_detalhadas
        GROUP BY cliente_id, cliente, cidade
        ORDER BY faturamento DESC;
    """
}

etapas["vendas_por_dia"] = {
    "tipo": "transformacao",
    "depende_de": ["vendas_detalhadas"],
    "descricao": "Quantidade de pedidos e faturamento por dia",
    "sql": """
        DROP TABLE IF EXISTS vendas_por_dia;
        CREATE TABLE vendas_por_dia AS
        SELECT data_pedido,
               COUNT(*) AS total_pedidos,
               SUM(valor_total) AS faturamento
        FROM vendas_detalhadas
        GROUP BY data_pedido
        ORDER BY data_pedido;
    """
}

# 4. Resultado final
etapas["resumo_vendas"] = {
    "tipo": "transformacao",
    "depende_de": ["vendas_detalhadas", "vendas_por_categoria", "vendas_por_cliente", "vendas_por_dia"],
    "descricao": "Indicadores finais: total de pedidos, faturamento, ticket médio, melhor categoria, cliente e dia",
    "sql": """
        DROP TABLE IF EXISTS resumo_vendas;
        CREATE TABLE resumo_vendas AS
        SELECT
            (SELECT COUNT(*) FROM vendas_detalhadas) AS total_pedidos,
            (SELECT SUM(valor_total) FROM vendas_detalhadas) AS faturamento_total,
            (SELECT ROUND(AVG(valor_total), 2) FROM vendas_detalhadas) AS ticket_medio,
            (SELECT nome_categoria FROM vendas_por_categoria ORDER BY faturamento DESC LIMIT 1) AS melhor_categoria,
            (SELECT cliente FROM vendas_por_cliente ORDER BY faturamento DESC LIMIT 1) AS melhor_cliente,
            (SELECT data_pedido FROM vendas_por_dia ORDER BY faturamento DESC LIMIT 1) AS melhor_dia;
    """
}

print(len(etapas), "etapas definidas")

------------------------------------------------------------------------------------------------------------
# **Construção do DAG**

In [ ]:
# 5. Construção do grafo direcionado com NetworkX
dag = nx.DiGraph()

for nome, etapa in etapas.items():
    dag.add_node(nome, tipo=etapa["tipo"])
    for dependencia in etapa["depende_de"]:
        if dependencia not in etapas:
            raise Exception(f"{nome} depende de {dependencia}, que não existe")
        # a aresta vai da dependência para a etapa que precisa dela
        dag.add_edge(dependencia, nome)

print("Etapas (nós):", dag.number_of_nodes())
print("Dependências (arestas):", dag.number_of_edges())
print("É acíclico?", nx.is_directed_acyclic_graph(dag))

In [ ]:
# 6. Ordem de execução (ordenação topológica)
ordem = list(nx.topological_sort(dag))

for i, nome in enumerate(ordem, start=1):
    deps = ", ".join(etapas[nome]["depende_de"]) or "-"
    print(f"{i:2}. {nome:22} depende de: {deps}")

# etapas do mesmo nível não dependem uma da outra, então poderiam rodar em paralelo
print()
for nivel, grupo in enumerate(nx.topological_generations(dag)):
    for nome in grupo:
        dag.nodes[nome]["nivel"] = nivel
    print(f"Nível {nivel}: {', '.join(grupo)}")

In [ ]:
# 7. Teste: o que acontece se alguém criar uma dependência circular
teste = dag.copy()
teste.add_edge("resumo_vendas", "pedidos_validos")

print("É acíclico?", nx.is_directed_acyclic_graph(teste))
print("Ciclo encontrado:", nx.find_cycle(teste))

------------------------------------------------------------------------------------------------------------
# **Execução do Pipeline**

As etapas rodam na ordem topológica. Se uma etapa falhar, tudo o que depende dela fica bloqueado e não é executado, o resto continua normalmente.

In [ ]:
# 8. Função que executa o pipeline seguindo o DAG
def executar_pipeline(etapas, dag, banco):
    conn = sqlite3.connect(banco)
    status = {}
    log = []

    for nome in nx.topological_sort(dag):
        etapa = etapas[nome]
        inicio = time.time()
        mensagem = ""

        # se alguma dependência não terminou com sucesso, a etapa fica bloqueada
        deps_com_problema = [d for d in etapa["depende_de"] if status[d] != "ok"]

        if deps_com_problema:
            status[nome] = "bloqueada"
            mensagem = "esperando " + ", ".join(deps_com_problema)
        else:
            try:
                if etapa["tipo"] == "verificacao":
                    problemas = pd.read_sql(etapa["sql"], conn)
                    if len(problemas) > 0:
                        lista = problemas["problema"] + " (pedido " + problemas["pedido_id"].astype(str) + ")"
                        raise Exception(f"{len(problemas)} problema(s): " + "; ".join(lista))
                else:
                    conn.executescript(etapa["sql"])
                status[nome] = "ok"
            except Exception as erro:
                status[nome] = "falhou"
                mensagem = str(erro)

        tempo = round((time.time() - inicio) * 1000, 1)
        log.append({"etapa": nome, "status": status[nome], "tempo_ms": tempo, "mensagem": mensagem})
        print(f"[{status[nome]}] {nome} {mensagem}")

    conn.close()
    return status, pd.DataFrame(log)


status, log = executar_pipeline(etapas, dag, "analytics.db")
log

In [ ]:
# 9. Resultado das transformações
conn = sqlite3.connect("analytics.db")

print("Resumo final:")
display(pd.read_sql("SELECT * FROM resumo_vendas", conn))

print("Vendas por categoria:")
display(pd.read_sql("SELECT * FROM vendas_por_categoria", conn))

print("Vendas por cliente:")
display(pd.read_sql("SELECT * FROM vendas_por_cliente", conn))

print("Pedidos descartados na limpeza:")
display(pd.read_sql("SELECT * FROM pedidos WHERE pedido_id NOT IN (SELECT pedido_id FROM pedidos_validos)", conn))

conn.close()

In [ ]:
# 10. Simulando um erro nos dados: pedido com valor_total errado (2 x 80,00 = 160,00 e não 100,00)
etapas_com_erro = copy.deepcopy(etapas)
etapas_com_erro["pedidos"]["sql"] = etapas_com_erro["pedidos"]["sql"].replace(
    "(1014, 5, 102, '2026-01-22', 1, 1200.00);",
    "(1014, 5, 102, '2026-01-22', 1, 1200.00), (1015, 4, 101, '2026-01-23', 2, 100.00);"
)

status_erro, log_erro = executar_pipeline(etapas_com_erro, dag, "analytics_erro.db")
log_erro

------------------------------------------------------------------------------------------------------------
# **Visualização do DAG**

In [ ]:
# 11. Renderização estática com Matplotlib (cor de acordo com o resultado da execução)
cores_status = {
    "ok": "#4CAF50",
    "falhou": "#F44336",
    "bloqueada": "#BDBDBD"
}

def desenhar_dag(dag, status, titulo):
    pos = nx.multipartite_layout(dag, subset_key="nivel")
    cores = [cores_status[status[n]] for n in dag.nodes()]

    plt.figure(figsize=(18, 8))
    nx.draw(dag, pos, with_labels=True, node_color=cores, node_size=1800, font_size=8,
            font_weight="bold", edge_color="gray", arrowsize=15)
    plt.title(titulo, fontsize=14)
    plt.show()

desenhar_dag(dag, status, "DAG de execução - pipeline normal")
desenhar_dag(dag, status_erro, "DAG de execução - com erro nos dados")

In [ ]:
# 12. Versão interativa em HTML com Pyvis (atividade extra)
def gerar_html(dag, status, log, arquivo):
    tempos = dict(zip(log["etapa"], log["tempo_ms"]))
    net = Network(height="600px", width="100%", directed=True, cdn_resources="in_line")

    for nome in nx.topological_sort(dag):
        etapa = etapas[nome]
        texto = f"{etapa['descricao']}\nStatus: {status[nome]} ({tempos[nome]} ms)\n\n{etapa['sql'].strip()}"
        forma = "diamond" if etapa["tipo"] == "verificacao" else "box"
        net.add_node(nome, label=nome, title=texto, shape=forma,
                     color=cores_status[status[nome]], level=dag.nodes[nome]["nivel"])

    for origem, destino in dag.edges():
        net.add_edge(origem, destino)

    # layout hierárquico da esquerda para a direita, seguindo a ordem de execução
    net.set_options("""
    {
      "layout": {"hierarchical": {"enabled": true, "direction": "LR", "sortMethod": "directed", "levelSeparation": 230}},
      "physics": {"enabled": false},
      "edges": {"arrows": {"to": {"enabled": true}}, "color": "gray"}
    }
    """)
    net.save_graph(arquivo)

gerar_html(dag, status, log, "dag_execucao.html")
gerar_html(dag, status_erro, log_erro, "dag_execucao_erro.html")

IPython.display.HTML(filename="dag_execucao.html")

------------------------------------------------------------------------------------------------------------
# **Explicação das Etapas**

| Etapa | Depende de | Responsabilidade |
|---|---|---|
| clientes, categorias, produtos, pedidos | - | Carga das tabelas de origem (mesma estrutura do diagrama do N1). |
| produtos_completos | produtos, categorias | Junta cada produto com sua categoria e departamento. |
| pedidos_validos | pedidos, clientes, produtos | Tira pedidos com quantidade zerada ou com cliente/produto inexistente (no exemplo saem o 1008 e o 1010). |
| verificacao_pedidos | pedidos_validos, produtos | Confere se o valor_total bate com quantidade x preço e se não há pedido duplicado. Se encontrar problema, falha e bloqueia o resto. |
| vendas_detalhadas | pedidos_validos, clientes, produtos_completos, verificacao_pedidos | Junta tudo em uma tabela única de vendas. |
| vendas_por_categoria | vendas_detalhadas | Faturamento e unidades por categoria. |
| vendas_por_cliente | vendas_detalhadas | Pedidos, faturamento e ticket médio por cliente. |
| vendas_por_dia | vendas_detalhadas | Pedidos e faturamento por dia. |
| resumo_vendas | vendas_detalhadas e as 3 agregações | Indicadores finais do pipeline. |

**Como o grafo coordena as transformações:**
- os dados entram pelas 4 tabelas de origem, passam pela limpeza e pela verificação, viram uma tabela única de vendas e terminam nas agregações e no resumo;
- a ordem de execução vem da ordenação topológica (`nx.topological_sort`), então uma etapa nunca roda antes das que ela depende;
- `nx.is_directed_acyclic_graph` garante que não existe ciclo (o teste do passo 7 mostra que um ciclo é detectado);
- a etapa de verificação funciona como uma trava: com o pedido 1015 errado ela falha e as etapas seguintes ficam bloqueadas, então nenhum indicador é gerado com dado inconsistente;
- todas as etapas usam `DROP TABLE IF EXISTS` + `CREATE TABLE`, então dá para rodar o pipeline de novo sem duplicar dados.